# N1 — Identificación y extracción de metáforas primarias (Procesamiento)
## AI-MELT: Nivel 1 — Metáfora primaria (versión 4 enfoques)

**Propósito:** Identificar expresiones metafóricas en cada oración del corpus y extraer los campos del modelo de datos del Nivel 1 de MELT, comparando cuatro enfoques técnicos.

**Entrada:** `n0_corpus.csv` (salida de N0)

**Salida:**
- Un CSV por enfoque: `n1_metaforas_{enfoque}.csv`
- Consolidados: `n1_metaforas_primarias.csv`, `n1_correspondencias_ontologicas.csv`, `n1_correspondencias_epistemicas.csv`
- Análisis: `n1_comparacion_enfoques.csv`, `n1_metadata.json`
- Muestra para evaluación: `n1_muestra_evaluacion.csv`

**Visualización:** Ejecutar `N1_metafora_primaria_viz.ipynb` después de este notebook.

---

### Enfoques
- **A — Claude API:** Chain-of-Thought con MIPVU
- **B — OpenAI API:** Structured outputs
- **C — HuggingFace:** Clasificación token-level cross-lingual
- **D — Embeddings:** MIP/SPV computacional con SentenceTransformers
- **E — Groq/Llama 3.1:** LLM abierto (8B) vía API gratuita de Groq, mismo prompt MIPVU
- **F — Gemini Flash:** LLM de Google vía API gratuita, mismo prompt MIPVU

## 1. Configuración

### 1.1 Instalación de dependencias

In [ ]:
# ============================================================
# 1.1 DEPENDENCIAS
# ============================================================
# Descomenta si necesitas instalar:
# !pip install anthropic openai pandas tqdm matplotlib seaborn wordcloud
# !pip install transformers torch sentence-transformers plotly scikit-learn
# !pip install python-dotenv
# !pip install groq google-genai

import importlib
required = ["anthropic", "openai", "pandas", "tqdm", "matplotlib", "seaborn",
            "transformers", "torch", "sentence_transformers", "plotly", "sklearn",
            "dotenv", "groq", "google.genai"]
missing = []
for pkg in required:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)
if missing:
    print(f"⚠ Paquetes faltantes: {missing}")
    print("  Ejecuta: pip install " + " ".join(missing)
          .replace("sklearn", "scikit-learn")
          .replace("sentence_transformers", "sentence-transformers")
          .replace("dotenv", "python-dotenv")
          .replace("google.genai", "google-genai"))
    raise ImportError(f"Falta instalar: {missing}")
print("✓ Todas las dependencias disponibles")

### 1.2 Imports

In [ ]:
# ============================================================
# 1.2 IMPORTS
# ============================================================
import anthropic
import openai
import pandas as pd
import numpy as np
import json
import os
import re
import time
from pathlib import Path
from tqdm.auto import tqdm
from collections import Counter, defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import plotly.graph_objects as go
import plotly.io as pio
from sklearn.metrics import cohen_kappa_score
import warnings
from dotenv import load_dotenv
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11
sns.set_style("whitegrid")

# Imports diferidos (para no cargar a menos que el enfoque esté activo)
# torch, transformers, sentence_transformers se importan en sus secciones

# groq y google.generativeai se importan en sus secciones (imports diferidos)

print("✓ Imports completados")

### 1.3 Configuración de enfoques y rutas

**Variables clave para controlar qué se ejecuta y qué se visualiza:**

- `ENFOQUES_ACTIVOS` — qué enfoques ejecutar en esta corrida del notebook
- `ENFOQUES_A_VISUALIZAR` — qué enfoques incluir en las visualizaciones consolidadas (pueden ser un subconjunto de los ejecutados, o incluir otros que ya estén en disco de corridas anteriores)
- `STANDALONE_MODE` — si los enfoques HF y embeddings operan independientes (True) o como pre-filtros del LLM (False)

In [ ]:
# ============================================================
# 1.3 CONFIGURACIÓN PRINCIPAL
# ============================================================

# ┌─────────────────────────────────────────────────────────┐
# │  MODIFICA ESTA SECCIÓN SEGÚN TUS NECESIDADES            │
# └─────────────────────────────────────────────────────────┘

# --- Qué enfoques EJECUTAR en esta corrida ---
# Quita los que no quieras correr:
# ENFOQUES_ACTIVOS = ["claude", "openai", "huggingface", "embeddings", "groq_llama", "gemini"]
ENFOQUES_ACTIVOS = ["claude", "openai"]

# --- Qué enfoques VISUALIZAR en las gráficas consolidadas ---
# Puede ser un subconjunto de los ejecutados o incluir otros ya guardados en disco
# ENFOQUES_A_VISUALIZAR = ["claude", "openai", "huggingface", "embeddings", "groq_llama", "gemini"]
ENFOQUES_A_VISUALIZAR = ["claude", "openai"]

# --- Modo de operación para C (HF) y D (embeddings) ---
# True  = standalone (salida propia con campos parciales)
# False = híbrido (pre-filtro + LLM extrae componentes)
STANDALONE_MODE = True

# Si es híbrido, qué LLM usar para la extracción de componentes
HYBRID_LLM = "openai"  # "claude" o "openai"

# --- API Keys ---
load_dotenv()
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")
OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not ANTHROPIC_API_KEY:
    print("⚠ ANTHROPIC_API_KEY no encontrada. Crea un archivo .env con: ANTHROPIC_API_KEY=sk-ant-...")
if not OPENAI_API_KEY:
    print("⚠ OPENAI_API_KEY no encontrada. Crea un archivo .env con: OPENAI_API_KEY=sk-...")
if not GROQ_API_KEY:
    print("⚠ GROQ_API_KEY no encontrada. Crea un archivo .env con: GROQ_API_KEY=gsk_...")
if not GEMINI_API_KEY:
    print("⚠ GEMINI_API_KEY no encontrada. Crea un archivo .env con: GEMINI_API_KEY=...")

# --- Rutas ---
INPUT_DIR = "./outputs/N0/"
OUTPUT_DIR = "./outputs/N1/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- Modelos LLM ---
CLAUDE_MODEL = "claude-sonnet-4-5"
OPENAI_MODEL = "gpt-4.1-mini"

# --- Modelos LLM adicionales (gratuitos) ---
GROQ_MODEL = "llama-3.1-8b-instant"      # Llama 3.1 8B vía Groq (gratuito)
GEMINI_MODEL = "gemini-2.0-flash"         # Gemini Flash vía Google (gratuito)

# --- Modelos HuggingFace y embeddings ---
HF_MODEL = "HiTZ/mdeberta-base-metaphor-detection-es"
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"

# --- Parámetros de procesamiento ---
TEMPERATURE = 0.1
MAX_RETRIES = 3
RATE_LIMIT_PAUSE = 1.0
SAMPLE_MODE = True # Parámetro en False cuando se vaya a recorrer todo el documento
SAMPLE_SIZE = 50

# --- Parámetros específicos del enfoque D (embeddings) ---
EMBEDDING_CONTRAST_THRESHOLD = 0.35  # Distancia coseno mínima para marcar como candidato
TARGET_POS = {"VERB", "NOUN", "ADJ"}  # POS que se evalúan

# --- Mapeo de colores por enfoque (consistente en todo el notebook) ---
COLORES_ENFOQUES = {
    "claude": "#3B8BD4",        # azul
    "openai": "#D85A30",        # naranja
    "huggingface": "#1D9E75",   # verde
    "embeddings": "#7F77DD",    # morado
    "groq_llama": "#E8A838",     # amarillo
    "gemini": "#FF6B8A",          # rosa
    # Añadir aquí nuevos enfoques si se agregan a futuro
}

# ID counter compartido entre enfoques para trazabilidad
GLOBAL_ID_COUNTER = {"claude": 0, "openai": 0, "huggingface": 0, "embeddings": 0, "groq_llama": 0, "gemini": 0}

def next_id(enfoque):
    GLOBAL_ID_COUNTER[enfoque] += 1
    prefix = enfoque[:3].upper()
    return f"M-{prefix}-{GLOBAL_ID_COUNTER[enfoque]:05d}"

def extract_expression(sentence, focus_token, focus_start, focus_end, window=5):
    """Extrae la expresión metafórica: el segmento de texto que contiene el foco.
    Toma una ventana de palabras alrededor del foco dentro de la oración.
    """
    words = sentence.split()
    # Encontrar la posición del foco en la lista de palabras
    char_pos = 0
    focus_word_idx = None
    for i, w in enumerate(words):
        if char_pos >= focus_start and char_pos <= focus_end:
            focus_word_idx = i
            break
        char_pos += len(w) + 1  # +1 por el espacio

    if focus_word_idx is None:
        # Fallback: buscar la palabra por texto
        for i, w in enumerate(words):
            if focus_token.lower() in w.lower():
                focus_word_idx = i
                break

    if focus_word_idx is None:
        return focus_token  # No se encontró, devolver solo el foco

    # Ventana de palabras alrededor del foco
    start = max(0, focus_word_idx - window)
    end = min(len(words), focus_word_idx + window + 1)
    expression = " ".join(words[start:end])
    return expression

# Inicializar resultados vacíos (se llenan en las celdas de cada enfoque)
results_claude = []
results_openai = []
results_huggingface = []
results_embeddings = []
results_groq_llama = []
results_gemini = []

# Tracking de tiempos y costos
tiempos_enfoques = {}
costos_enfoques = {}
tokens_enfoques = {}

print(f"✓ Configuración cargada")
print(f"  Enfoques ACTIVOS (ejecutar):    {ENFOQUES_ACTIVOS}")
print(f"  Enfoques a VISUALIZAR:          {ENFOQUES_A_VISUALIZAR}")
print(f"  Modo:                           {'STANDALONE' if STANDALONE_MODE else 'HÍBRIDO (pre-filtro + ' + HYBRID_LLM + ')'}")
print(f"  Sample:                         {SAMPLE_SIZE if SAMPLE_MODE else 'corpus completo'}")

## 2. Carga de datos N0

In [ ]:
# ============================================================
# 2. CARGA DE DATOS N0
# ============================================================

input_path = os.path.join(INPUT_DIR, "n0_corpus.csv")
df_n0 = pd.read_csv(input_path)
print(f"✓ Cargado: {input_path}")
print(f"  Oraciones: {len(df_n0):,}")
print(f"  Documentos: {df_n0['ID_documento'].nunique()}")

# Seleccionar muestra o corpus completo
if SAMPLE_MODE:
    df_work = df_n0.sample(n=min(SAMPLE_SIZE, len(df_n0)), random_state=42).reset_index(drop=True)
    print(f"\n⚠ MODO MUESTRA: procesando {len(df_work)} oraciones")
else:
    df_work = df_n0.copy().reset_index(drop=True)
    print(f"\n📊 CORPUS COMPLETO: procesando {len(df_work):,} oraciones")

# Construir columna de contexto ampliado
df_work = df_work.sort_values(["ID_documento", "ID_oracion"]).reset_index(drop=True)

contexts = []
for i, row in df_work.iterrows():
    prev_sent = df_work.iloc[i-1]["oracion_texto"] if i > 0 and df_work.iloc[i-1]["ID_documento"] == row["ID_documento"] else ""
    next_sent = df_work.iloc[i+1]["oracion_texto"] if i < len(df_work)-1 and df_work.iloc[i+1]["ID_documento"] == row["ID_documento"] else ""
    contexts.append(f"{prev_sent} ||| {row['oracion_texto']} ||| {next_sent}")

df_work["contexto_ampliado"] = contexts
print(f"\nContextos construidos: {len(contexts)}")


## 3. Diseño del prompt MIPVU

El prompt se usa tanto en Claude como en OpenAI, y también en modo híbrido cuando HF/embeddings delegan al LLM.

In [ ]:
# ============================================================
# 3. PROMPT MIPVU COMPARTIDO
# ============================================================

SYSTEM_PROMPT = """Eres un lingüista cognitivo experto en la Teoría de la Metáfora Conceptual (Lakoff y Johnson, 1980) y en el procedimiento MIPVU (Steen et al., 2010) para la identificación de metáforas. Tu tarea es analizar oraciones del Informe Final de la Comisión de la Verdad de Colombia e identificar expresiones metafóricas.

## Procedimiento MIPVU que debes seguir:
1. Lee la oración completa y su contexto
2. Para cada unidad léxica de contenido (sustantivos, verbos, adjetivos, adverbios):
   a. Determina su SIGNIFICADO CONTEXTUAL: el sentido que la palabra adquiere en esta oración específica
   b. Determina su SIGNIFICADO BÁSICO: el sentido más concreto, más corporal, más preciso o históricamente anterior
   c. Si existe un CONTRASTE entre ambos significados, y el contextual puede explicarse mediante una transferencia desde el básico, la expresión es METAFÓRICA
3. Para cada expresión metafórica identificada, extrae todos los componentes analíticos

## Criterios importantes:
- Las metáforas convencionales (lexicalizadas) TAMBIÉN cuentan: "caer en la violencia", "construir la paz"
- Las personificaciones SON metáforas: "la violencia habla", "el país necesita"
- NO marques como metáfora: expresiones literales, metonimias puras, modismos sin base metafórica
- Si no hay metáforas en la oración, devuelve una lista vacía

## Formato de salida (JSON estricto):
Responde SOLO con un JSON válido, sin texto adicional, sin markdown, sin backticks.
"""

USER_PROMPT_TEMPLATE = """Analiza la siguiente oración y su contexto. Identifica TODAS las expresiones metafóricas.

CONTEXTO (oración anterior ||| oración actual ||| oración siguiente):
{contexto}

ORACIÓN A ANALIZAR:
{oracion}

{candidatos_info}

Para cada metáfora encontrada, devuelve un JSON con esta estructura exacta:
{{
  "metaforas": [
    {{
      "expresion_metaforica": "la frase metafórica tal como aparece en el texto",
      "foco": "la palabra que porta el sentido metafórico",
      "foco_lemma": "lema del foco",
      "foco_part_of_speech": "VERB|NOUN|ADJ|ADV",
      "significado_contextual": "sentido de la palabra en esta oración",
      "significado_basico": "sentido más concreto, corporal o históricamente anterior",
      "dominio_fuente": "DOMINIO FUENTE en mayúsculas (el concreto)",
      "dominio_meta": "DOMINIO META en mayúsculas (el abstracto)",
      "metafora_conceptual": "DOMINIO META ES DOMINIO FUENTE",
      "correspondencias_ontologicas": [
        {{
          "elemento_fuente": "entidad/propiedad en el dominio fuente",
          "elemento_meta": "entidad/propiedad en el dominio meta",
          "evidencia_textual": "fragmento del texto que sustenta este mapeo"
        }}
      ],
      "correspondencias_epistemicas": [
        {{
          "tipo_inferencia": "CAUSAL|TEMPORAL|CONDICIONAL|NORMATIVA|EVALUATIVA",
          "relacion_fuente": "conocimiento inferencial en el dominio fuente",
          "inferencia_meta": "conocimiento transferido al dominio meta",
          "evidencia_textual": "fragmento del texto"
        }}
      ]
    }}
  ]
}}

Si no hay metáforas en la oración, responde: {{"metaforas": []}}
"""

FEW_SHOT_EXAMPLES = [
    {
        "oracion": "El conflicto armado ha dejado heridas profundas en el tejido social colombiano.",
        "respuesta": {
            "metaforas": [
                {
                    "expresion_metaforica": "heridas profundas en el tejido social",
                    "foco": "heridas",
                    "foco_lemma": "herida",
                    "foco_part_of_speech": "NOUN",
                    "significado_contextual": "daños emocionales, traumas colectivos causados por el conflicto",
                    "significado_basico": "lesión física en el cuerpo, corte o abertura en la piel",
                    "dominio_fuente": "CUERPO FÍSICO",
                    "dominio_meta": "SOCIEDAD",
                    "metafora_conceptual": "LA SOCIEDAD ES UN CUERPO",
                    "correspondencias_ontologicas": [
                        {"elemento_fuente": "herida en el cuerpo", "elemento_meta": "trauma colectivo", "evidencia_textual": "heridas profundas"},
                        {"elemento_fuente": "tejido corporal", "elemento_meta": "estructura social", "evidencia_textual": "tejido social"}
                    ],
                    "correspondencias_epistemicas": [
                        {"tipo_inferencia": "CAUSAL", "relacion_fuente": "Las heridas causan dolor y debilitan el cuerpo", "inferencia_meta": "Los traumas causan sufrimiento y debilitan la sociedad", "evidencia_textual": "heridas profundas en el tejido social"},
                        {"tipo_inferencia": "TEMPORAL", "relacion_fuente": "Las heridas requieren tiempo para sanar", "inferencia_meta": "Los traumas sociales requieren tiempo para superarse", "evidencia_textual": "ha dejado heridas"}
                    ]
                }
            ]
        }
    }
]

print(f"✓ Prompt MIPVU diseñado")
print(f"  System prompt: {len(SYSTEM_PROMPT)} caracteres")
print(f"  Few-shot examples: {len(FEW_SHOT_EXAMPLES)}")

## 4. Enfoque A — Claude API

Solo se ejecuta si `"claude"` está en `ENFOQUES_ACTIVOS`.

In [ ]:
# ============================================================
# 4. ENFOQUE A: CLAUDE
# ============================================================

def llm_claude(oracion, contexto, candidatos_info=""):
    """Llama a Claude con el prompt MIPVU. Devuelve (resultado, tokens_in, tokens_out)."""
    messages = []
    for ex in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user", "content": USER_PROMPT_TEMPLATE.format(
            oracion=ex["oracion"], contexto=f"||| {ex['oracion']} |||", candidatos_info="")})
        messages.append({"role": "assistant", "content": json.dumps(ex["respuesta"], ensure_ascii=False)})
    messages.append({"role": "user", "content": USER_PROMPT_TEMPLATE.format(
        oracion=oracion, contexto=contexto, candidatos_info=candidatos_info)})

    for attempt in range(MAX_RETRIES):
        try:
            response = client_claude.messages.create(
            model=CLAUDE_MODEL, max_tokens=4096, temperature=TEMPERATURE,
            system=[{
                "type": "text",
                "text": SYSTEM_PROMPT,
                "cache_control": {"type": "ephemeral"}
            }],
            messages=messages)
            text = response.content[0].text.strip()
            text = re.sub(r'^```json\s*', '', text)
            text = re.sub(r'\s*```$', '', text)
            return json.loads(text), response.usage.input_tokens, response.usage.output_tokens
        except (json.JSONDecodeError, Exception) as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(2 * (attempt + 1))
            else:
                print(f"  ⚠ Error Claude: {e}")
    return {"metaforas": []}, 0, 0


if "claude" in ENFOQUES_ACTIVOS:
    client_claude = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    results_claude = []
    tok_in_total = 0
    tok_out_total = 0
    t0 = time.time()

    print(f"Procesando {len(df_work)} oraciones con Claude ({CLAUDE_MODEL})...")
    for idx, row in tqdm(df_work.iterrows(), total=len(df_work), desc="Claude"):
        result, ti, to = llm_claude(row["oracion_texto"], row["contexto_ampliado"])
        tok_in_total += ti
        tok_out_total += to
        for met in result.get("metaforas", []):
            met["ID_expresion"] = next_id("claude")
            met["ID_oracion"] = row["ID_oracion"]
            met["ID_documento"] = row["ID_documento"]
            met["pagina"] = row.get("pagina", "")
            met["contexto"] = row["oracion_texto"]
            met["enfoque"] = "claude"
            met["confianza_modelo"] = 1.0  # LLM no provee score explícito
            results_claude.append(met)
        time.sleep(RATE_LIMIT_PAUSE)

    tiempos_enfoques["claude"] = time.time() - t0
    tokens_enfoques["claude"] = {"in": tok_in_total, "out": tok_out_total}
    costos_enfoques["claude"] = (tok_in_total / 1e6 * 3) + (tok_out_total / 1e6 * 15)

    print(f"\n✓ Claude completado")
    print(f"  Metáforas: {len(results_claude)}")
    print(f"  Tiempo: {tiempos_enfoques['claude']:.1f}s")
    print(f"  Tokens: {tok_in_total:,} in + {tok_out_total:,} out")
    print(f"  Costo: ${costos_enfoques['claude']:.2f} USD")
else:
    print("⏭ Enfoque Claude omitido (no está en ENFOQUES_ACTIVOS)")
    results_claude = []

## 5. Enfoque B — OpenAI API

Solo se ejecuta si `"openai"` está en `ENFOQUES_ACTIVOS`.

In [ ]:
# ============================================================
# 5. ENFOQUE B: OPENAI
# ============================================================

def llm_openai(oracion, contexto, candidatos_info=""):
    """Llama a OpenAI con el prompt MIPVU. Devuelve (resultado, tokens_in, tokens_out)."""
    messages = [{"role": "system", "content": SYSTEM_PROMPT}]
    for ex in FEW_SHOT_EXAMPLES:
        messages.append({"role": "user", "content": USER_PROMPT_TEMPLATE.format(
            oracion=ex["oracion"], contexto=f"||| {ex['oracion']} |||", candidatos_info="")})
        messages.append({"role": "assistant", "content": json.dumps(ex["respuesta"], ensure_ascii=False)})
    messages.append({"role": "user", "content": USER_PROMPT_TEMPLATE.format(
        oracion=oracion, contexto=contexto, candidatos_info=candidatos_info)})

    for attempt in range(MAX_RETRIES):
        try:
            response = client_openai.chat.completions.create(
                model=OPENAI_MODEL, messages=messages, temperature=TEMPERATURE,
                max_tokens=4096, response_format={"type": "json_object"})
            return (json.loads(response.choices[0].message.content.strip()),
                    response.usage.prompt_tokens, response.usage.completion_tokens)
        except (json.JSONDecodeError, Exception) as e:
            if attempt < MAX_RETRIES - 1:
                time.sleep(2 * (attempt + 1))
            else:
                print(f"  ⚠ Error OpenAI: {e}")
    return {"metaforas": []}, 0, 0


if "openai" in ENFOQUES_ACTIVOS:
    client_openai = openai.OpenAI(api_key=OPENAI_API_KEY)
    results_openai = []
    tok_in_total = 0
    tok_out_total = 0
    t0 = time.time()

    print(f"Procesando {len(df_work)} oraciones con OpenAI ({OPENAI_MODEL})...")
    for idx, row in tqdm(df_work.iterrows(), total=len(df_work), desc="OpenAI"):
        result, ti, to = llm_openai(row["oracion_texto"], row["contexto_ampliado"])
        tok_in_total += ti
        tok_out_total += to
        for met in result.get("metaforas", []):
            met["ID_expresion"] = next_id("openai")
            met["ID_oracion"] = row["ID_oracion"]
            met["ID_documento"] = row["ID_documento"]
            met["pagina"] = row.get("pagina", "")
            met["contexto"] = row["oracion_texto"]
            met["enfoque"] = "openai"
            met["confianza_modelo"] = 1.0
            results_openai.append(met)
        time.sleep(RATE_LIMIT_PAUSE)

    tiempos_enfoques["openai"] = time.time() - t0
    tokens_enfoques["openai"] = {"in": tok_in_total, "out": tok_out_total}
    costos_enfoques["openai"] = (tok_in_total / 1e6 * 2.5) + (tok_out_total / 1e6 * 10)

    print(f"\n✓ OpenAI completado")
    print(f"  Metáforas: {len(results_openai)}")
    print(f"  Tiempo: {tiempos_enfoques['openai']:.1f}s")
    print(f"  Costo: ${costos_enfoques['openai']:.2f} USD")
else:
    print("⏭ Enfoque OpenAI omitido (no está en ENFOQUES_ACTIVOS)")
    results_openai = []

## 6. Enfoque C — HuggingFace Transformer

Clasificación token-level con `joelito/xlm-roberta-large-english-text-metaphor-detection`. Aunque el modelo está entrenado en inglés, la base XLM-RoBERTa multilingüe permite transferencia cross-lingual al español (cf. Tsvetkov et al. 2014, Sánchez-Bayona y Agerri 2022).

- **Standalone:** Produce una fila por token metafórico con campos parciales (foco, POS, score)
- **Híbrido:** Las oraciones con tokens metafóricos se envían al LLM (`HYBRID_LLM`) para extracción completa de componentes

In [ ]:
# ============================================================
# 6. ENFOQUE C: HUGGINGFACE TRANSFORMER
# ============================================================

if "huggingface" in ENFOQUES_ACTIVOS:
    import torch
    from transformers import AutoTokenizer, AutoModelForTokenClassification

    print(f"Cargando modelo HuggingFace: {HF_MODEL}...")
    try:
        tokenizer_hf = AutoTokenizer.from_pretrained(HF_MODEL)
        model_hf = AutoModelForTokenClassification.from_pretrained(HF_MODEL)
        model_hf.eval()
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        model_hf = model_hf.to(device)
        print(f"  Device: {device}")

        # Detectar esquema de etiquetas del modelo
        id2label = model_hf.config.id2label
        print(f"  Etiquetas del modelo: {id2label}")

        # Identificar qué índices corresponden a metáfora
        # Soporta esquema BIO (B-METAPHOR, I-METAPHOR, O) y binario (0=literal, 1=metaphor)
        metaphor_indices = []
        for idx, label in id2label.items():
            label_upper = str(label).upper()
            if "METAPHOR" in label_upper or "MET" in label_upper:
                if "O" != label_upper:  # Excluir la etiqueta "O" (Outside)
                    metaphor_indices.append(int(idx))
        
        if not metaphor_indices:
            # Fallback: asumir que la última clase NO es metáfora (esquema binario)
            metaphor_indices = list(range(len(id2label) - 1))
        
        print(f"  Índices de metáfora: {metaphor_indices} → {[id2label[i] for i in metaphor_indices]}")

    except Exception as e:
        print(f"  ⚠ Error cargando modelo: {e}")
        print(f"  Verifica que el modelo esté disponible en HuggingFace Hub")
        raise

    def detect_metaphor_tokens_hf(sentence, threshold=0.5):
        """Clasifica tokens como metafóricos/literales.
        Suma las probabilidades de todas las clases de metáfora (B-METAPHOR + I-METAPHOR)
        para obtener el score de metaforicidad de cada token.
        """
        inputs = tokenizer_hf(sentence, return_tensors="pt", truncation=True,
                              max_length=512, return_offsets_mapping=True)
        offsets = inputs.pop("offset_mapping")[0].tolist()
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model_hf(**inputs)
            probs = torch.softmax(outputs.logits, dim=-1)[0].cpu().numpy()

        tokens = tokenizer_hf.convert_ids_to_tokens(inputs["input_ids"][0].cpu().numpy())
        metaphorical = []

        for i, (tok, p, off) in enumerate(zip(tokens, probs, offsets)):
            if tok in tokenizer_hf.all_special_tokens:
                continue
            # Sumar probabilidades de todas las clases de metáfora
            score = sum(float(p[idx]) for idx in metaphor_indices)
            if score >= threshold:
                word = sentence[off[0]:off[1]]
                if word.strip():
                    metaphorical.append({"token": word, "score": score, "start": off[0], "end": off[1]})
        return metaphorical

    results_huggingface = []
    t0 = time.time()

    print(f"Procesando {len(df_work)} oraciones con HuggingFace...")
    for idx, row in tqdm(df_work.iterrows(), total=len(df_work), desc="HuggingFace"):
        sentence = row["oracion_texto"]
        try:
            metaphorical_tokens = detect_metaphor_tokens_hf(sentence, threshold=0.5)
        except Exception as e:
            metaphorical_tokens = []

        if not metaphorical_tokens:
            continue

        if STANDALONE_MODE:
            for mt in metaphorical_tokens:
                met = {
                    "ID_expresion": next_id("huggingface"),
                    "ID_oracion": row["ID_oracion"],
                    "ID_documento": row["ID_documento"],
                    "pagina": row.get("pagina", ""),
                    "contexto": sentence,
                    "expresion_metaforica": extract_expression(sentence, mt["token"], mt["start"], mt["end"]),
                    "foco": mt["token"],
                    "foco_lemma": "",
                    "foco_part_of_speech": "",
                    "significado_contextual": "",
                    "significado_basico": "",
                    "dominio_fuente": "",
                    "dominio_meta": "",
                    "metafora_conceptual": "",
                    "correspondencias_ontologicas": [],
                    "correspondencias_epistemicas": [],
                    "enfoque": "huggingface",
                    "confianza_modelo": mt["score"]
                }
                results_huggingface.append(met)
        else:
            tokens_desc = ", ".join([f"'{mt['token']}' ({mt['score']:.2f})" for mt in metaphorical_tokens])
            candidatos_info = (f"PRE-FILTRO: El modelo HuggingFace marcó estas palabras como "
                               f"potencialmente metafóricas (score): {tokens_desc}")
            if HYBRID_LLM == "claude":
                result, _, _ = llm_claude(sentence, row["contexto_ampliado"], candidatos_info)
            else:
                result, _, _ = llm_openai(sentence, row["contexto_ampliado"], candidatos_info)

            for met in result.get("metaforas", []):
                met["ID_expresion"] = next_id("huggingface")
                met["ID_oracion"] = row["ID_oracion"]
                met["ID_documento"] = row["ID_documento"]
                met["pagina"] = row.get("pagina", "")
                met["contexto"] = sentence
                met["enfoque"] = "huggingface"
                met["confianza_modelo"] = max(mt["score"] for mt in metaphorical_tokens)
                results_huggingface.append(met)
            time.sleep(RATE_LIMIT_PAUSE)

    tiempos_enfoques["huggingface"] = time.time() - t0
    print(f"\n✓ HuggingFace completado")
    print(f"  Metáforas: {len(results_huggingface)}")
    print(f"  Tiempo: {tiempos_enfoques['huggingface']:.1f}s")
    print(f"  Modo: {'STANDALONE' if STANDALONE_MODE else 'HÍBRIDO con ' + HYBRID_LLM}")
    costos_enfoques["huggingface"] = 0.0
else:
    print("⏭ Enfoque HuggingFace omitido (no está en ENFOQUES_ACTIVOS)")
    results_huggingface = []

## 7. Enfoque D — Contraste de embeddings (MIP/SPV computacional)

Operacionaliza el procedimiento MIP comparando el embedding del foco en contexto (significado contextual) con el embedding del foco aislado (aproximación al significado básico). Una distancia coseno alta sugiere contraste semántico → candidato a metáfora.

**Base teórica:** Mao, Lin y Guerin (2018, 2019), Ottolina et al. (2022).

In [ ]:
# ============================================================
# 7. ENFOQUE D: CONTRASTE DE EMBEDDINGS
# ============================================================

if "embeddings" in ENFOQUES_ACTIVOS:
    from sentence_transformers import SentenceTransformer
    import spacy

    print(f"Cargando modelo de embeddings: {EMBEDDING_MODEL}...")
    model_emb = SentenceTransformer(EMBEDDING_MODEL)

    # spaCy para POS tagging y lematización
    try:
        nlp_es = spacy.load("es_core_news_lg")
    except OSError:
        print("  Descargando spaCy es_core_news_lg...")
        os.system("python -m spacy download es_core_news_lg")
        nlp_es = spacy.load("es_core_news_lg")

    def cosine_sim(a, b):
        return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-10))

    def detect_metaphor_embeddings(sentence, threshold=EMBEDDING_CONTRAST_THRESHOLD):
        """
        Para cada palabra de contenido (VERB, NOUN, ADJ):
        1. Embedding contextual: la palabra dentro de la oración completa (significado contextual)
        2. Embedding aislado: la palabra en contexto prototípico literal (significado básico)
        3. Si la distancia coseno supera el umbral -> candidato a metáfora
        """
        doc = nlp_es(sentence)
        candidates = []

        # Embedding de la oración completa
        sent_emb = model_emb.encode(sentence, convert_to_numpy=True, show_progress_bar=False)

        for token in doc:
            if token.pos_ not in TARGET_POS or token.is_stop or len(token.text) < 3:
                continue

            # Aproximación al significado básico: la palabra aislada
            # (SentenceTransformers devuelve un embedding agregado)
            basic_emb = model_emb.encode(token.lemma_, convert_to_numpy=True, show_progress_bar=False)

            # Aproximación al significado contextual: palabra en una plantilla simple
            # que extrae su uso en este contexto
            context_template = f"La palabra {token.text} en: {sentence}"
            context_emb = model_emb.encode(context_template, convert_to_numpy=True, show_progress_bar=False)

            # Distancia entre significado básico y contextual
            # (cuanto más bajo el coseno, mayor la distancia, mayor probabilidad de metáfora)
            similarity = cosine_sim(basic_emb, context_emb)
            distance = 1.0 - similarity

            if distance >= threshold:
                candidates.append({
                    "token": token.text,
                    "lemma": token.lemma_,
                    "pos": token.pos_,
                    "distance": distance,
                    "score": float(distance)  # Score normalizado a [0, 1]
                })

        return candidates

    results_embeddings = []
    t0 = time.time()

    print(f"Procesando {len(df_work)} oraciones con embeddings...")
    for idx, row in tqdm(df_work.iterrows(), total=len(df_work), desc="Embeddings"):
        sentence = row["oracion_texto"]
        try:
            candidates = detect_metaphor_embeddings(sentence)
        except Exception as e:
            candidates = []

        if not candidates:
            continue

        if STANDALONE_MODE:
            for c in candidates:
                met = {
                    "ID_expresion": next_id("embeddings"),
                    "ID_oracion": row["ID_oracion"],
                    "ID_documento": row["ID_documento"],
                    "pagina": row.get("pagina", ""),
                    "contexto": sentence,
                    "expresion_metaforica": extract_expression(sentence, c["token"], 0, len(c["token"])),
                    "foco": c["token"],
                    "foco_lemma": c["lemma"],
                    "foco_part_of_speech": c["pos"],
                    "significado_contextual": "",        # No inferible
                    "significado_basico": "",            # No inferible
                    "dominio_fuente": "",                # No inferible
                    "dominio_meta": "",                  # No inferible
                    "metafora_conceptual": "",           # No inferible
                    "correspondencias_ontologicas": [],
                    "correspondencias_epistemicas": [],
                    "enfoque": "embeddings",
                    "confianza_modelo": c["score"]
                }
                results_embeddings.append(met)
        else:
            # MODO HÍBRIDO: pasar candidatos al LLM
            tokens_desc = ", ".join([f"'{c['token']}' ({c['distance']:.2f})" for c in candidates])
            candidatos_info = (f"PRE-FILTRO: El análisis de embeddings marcó estas palabras "
                               f"como potencialmente metafóricas (distancia coseno): {tokens_desc}")
            if HYBRID_LLM == "claude":
                result, _, _ = llm_claude(sentence, row["contexto_ampliado"], candidatos_info)
            else:
                result, _, _ = llm_openai(sentence, row["contexto_ampliado"], candidatos_info)

            for met in result.get("metaforas", []):
                met["ID_expresion"] = next_id("embeddings")
                met["ID_oracion"] = row["ID_oracion"]
                met["ID_documento"] = row["ID_documento"]
                met["pagina"] = row.get("pagina", "")
                met["contexto"] = sentence
                met["enfoque"] = "embeddings"
                met["confianza_modelo"] = max(c["score"] for c in candidates)
                results_embeddings.append(met)
            time.sleep(RATE_LIMIT_PAUSE)

    tiempos_enfoques["embeddings"] = time.time() - t0
    print(f"\n✓ Embeddings completado")
    print(f"  Metáforas: {len(results_embeddings)}")
    print(f"  Tiempo: {tiempos_enfoques['embeddings']:.1f}s")
    print(f"  Modo: {'STANDALONE' if STANDALONE_MODE else 'HÍBRIDO con ' + HYBRID_LLM}")
    costos_enfoques["embeddings"] = 0.0
else:
    print("⏭ Enfoque embeddings omitido (no está en ENFOQUES_ACTIVOS)")
    results_embeddings = []

## 8. Enfoque E — Groq API (Llama 3.1 8B)

LLM abierto vía API gratuita de Groq. Usa el mismo prompt MIPVU que los enfoques A y B, lo que permite comparar un modelo abierto de 8B parámetros contra Claude y GPT con idéntica instrucción.

**Base teórica:** Tian et al. (2024) TSI, Puraivan et al. (2024).

In [ ]:
# ============================================================
# 8. ENFOQUE E: GROQ API (LLAMA 3.1 8B)
# ============================================================

if "groq_llama" in ENFOQUES_ACTIVOS:
    from groq import Groq

    client_groq = Groq(api_key=GROQ_API_KEY)

    def llm_groq(oracion, contexto, candidatos_info=""):
        """Llama a Groq (Llama 3.1) con el prompt MIPVU."""
        messages = [{"role": "system", "content": SYSTEM_PROMPT}]
        for ex in FEW_SHOT_EXAMPLES:
            messages.append({"role": "user", "content": USER_PROMPT_TEMPLATE.format(
                oracion=ex["oracion"], contexto=f"||| {ex['oracion']} |||", candidatos_info="")})
            messages.append({"role": "assistant", "content": json.dumps(ex["respuesta"], ensure_ascii=False)})
        messages.append({"role": "user", "content": USER_PROMPT_TEMPLATE.format(
            oracion=oracion, contexto=contexto, candidatos_info=candidatos_info)})

        for attempt in range(MAX_RETRIES):
            try:
                response = client_groq.chat.completions.create(
                    model=GROQ_MODEL, messages=messages,
                    temperature=TEMPERATURE, max_tokens=4096,
                    response_format={"type": "json_object"})
                text = response.choices[0].message.content.strip()
                text = re.sub(r'^```json\s*', '', text)
                text = re.sub(r'\s*```$', '', text)
                tok_in = response.usage.prompt_tokens
                tok_out = response.usage.completion_tokens
                return json.loads(text), tok_in, tok_out
            except (json.JSONDecodeError, Exception) as e:
                if attempt < MAX_RETRIES - 1:
                    time.sleep(2 * (attempt + 1))
                else:
                    print(f"  ⚠ Error Groq: {e}")
        return {"metaforas": []}, 0, 0

    results_groq_llama = []
    tok_in_total = 0
    tok_out_total = 0
    t0 = time.time()

    print(f"Procesando {len(df_work)} oraciones con Groq/Llama ({GROQ_MODEL})...")
    for idx, row in tqdm(df_work.iterrows(), total=len(df_work), desc="Groq/Llama"):
        result, ti, to = llm_groq(row["oracion_texto"], row["contexto_ampliado"])
        tok_in_total += ti
        tok_out_total += to
        for met in result.get("metaforas", []):
            met["ID_expresion"] = next_id("groq_llama")
            met["ID_oracion"] = row["ID_oracion"]
            met["ID_documento"] = row["ID_documento"]
            met["pagina"] = row.get("pagina", "")
            met["contexto"] = row["oracion_texto"]
            met["enfoque"] = "groq_llama"
            met["confianza_modelo"] = 1.0
            results_groq_llama.append(met)
        time.sleep(RATE_LIMIT_PAUSE)

    tiempos_enfoques["groq_llama"] = time.time() - t0
    tokens_enfoques["groq_llama"] = {"in": tok_in_total, "out": tok_out_total}
    costos_enfoques["groq_llama"] = 0.0  # Groq free tier

    print(f"\n✓ Groq/Llama completado")
    print(f"  Metáforas: {len(results_groq_llama)}")
    print(f"  Tiempo: {tiempos_enfoques['groq_llama']:.1f}s")
    print(f"  Tokens: {tok_in_total:,} in + {tok_out_total:,} out")
    print(f"  Costo: $0.00 USD (free tier)")
else:
    print("⏭ Enfoque Groq/Llama omitido (no está en ENFOQUES_ACTIVOS)")
    results_groq_llama = []

## 9. Enfoque F — Google Gemini Flash

LLM de Google con free tier generoso. Usa el mismo prompt MIPVU. Permite comparar un tercer proveedor de LLM contra Claude, GPT y Llama.

**Base teórica:** Mismo prompt MIPVU (Steen et al., 2010) aplicado a distinto modelo generativo.

In [ ]:
# ============================================================
# 9. ENFOQUE F: GOOGLE GEMINI FLASH
# ============================================================

if "gemini" in ENFOQUES_ACTIVOS:
    from google import genai

    client_gemini = genai.Client(api_key=GEMINI_API_KEY)

    def llm_gemini(oracion, contexto, candidatos_info=""):
        """Llama a Gemini Flash con el prompt MIPVU."""
        few_shot_text = ""
        for ex in FEW_SHOT_EXAMPLES:
            few_shot_text += "EJEMPLO:\n"
            few_shot_text += USER_PROMPT_TEMPLATE.format(
                oracion=ex["oracion"], contexto=f"||| {ex['oracion']} |||", candidatos_info="") + "\n"
            few_shot_text += "RESPUESTA:\n" + json.dumps(ex["respuesta"], ensure_ascii=False) + "\n\n"

        full_prompt = (SYSTEM_PROMPT + "\n\n" + few_shot_text +
                       "AHORA ANALIZA:\n" +
                       USER_PROMPT_TEMPLATE.format(
                           oracion=oracion, contexto=contexto, candidatos_info=candidatos_info))

        for attempt in range(MAX_RETRIES):
            try:
                response = client_gemini.models.generate_content(
                    model=GEMINI_MODEL,
                    contents=full_prompt,
                    config={
                        "temperature": TEMPERATURE,
                        "max_output_tokens": 4096,
                        "response_mime_type": "application/json"
                    })
                text = response.text.strip()
                text = re.sub(r'^```json\s*', '', text)
                text = re.sub(r'\s*```$', '', text)
                tok_in = getattr(response, 'usage_metadata', None)
                tok_in_count = tok_in.prompt_token_count if tok_in else 0
                tok_out_count = tok_in.candidates_token_count if tok_in else 0
                return json.loads(text), tok_in_count, tok_out_count
            except (json.JSONDecodeError, Exception) as e:
                if attempt < MAX_RETRIES - 1:
                    time.sleep(3 * (attempt + 1))
                else:
                    print(f"  ⚠ Error Gemini: {e}")
        return {"metaforas": []}, 0, 0

    results_gemini = []
    tok_in_total = 0
    tok_out_total = 0
    t0 = time.time()

    print(f"Procesando {len(df_work)} oraciones con Gemini ({GEMINI_MODEL})...")
    for idx, row in tqdm(df_work.iterrows(), total=len(df_work), desc="Gemini"):
        result, ti, to = llm_gemini(row["oracion_texto"], row["contexto_ampliado"])
        tok_in_total += ti
        tok_out_total += to
        for met in result.get("metaforas", []):
            met["ID_expresion"] = next_id("gemini")
            met["ID_oracion"] = row["ID_oracion"]
            met["ID_documento"] = row["ID_documento"]
            met["pagina"] = row.get("pagina", "")
            met["contexto"] = row["oracion_texto"]
            met["enfoque"] = "gemini"
            met["confianza_modelo"] = 1.0
            results_gemini.append(met)
        time.sleep(RATE_LIMIT_PAUSE)

    tiempos_enfoques["gemini"] = time.time() - t0
    tokens_enfoques["gemini"] = {"in": tok_in_total, "out": tok_out_total}
    costos_enfoques["gemini"] = 0.0  # Gemini free tier

    print(f"\n✓ Gemini completado")
    print(f"  Metáforas: {len(results_gemini)}")
    print(f"  Tiempo: {tiempos_enfoques['gemini']:.1f}s")
    print(f"  Tokens: {tok_in_total:,} in + {tok_out_total:,} out")
    print(f"  Costo: $0.00 USD (free tier)")
else:
    print("⏭ Enfoque Gemini omitido (no está en ENFOQUES_ACTIVOS)")
    results_gemini = []

## 10. Exportación de resultados por enfoque

Cada enfoque se exporta a su propio CSV. Esto permite que el resto del notebook se pueda ejecutar en sesiones posteriores leyendo solo desde disco.

In [ ]:
# ============================================================
# 10. EXPORTACIÓN DE RESULTADOS POR ENFOQUE
# ============================================================

def flatten_approach_results(results_list, enfoque_label):
    """Convierte resultados anidados en DataFrames planos."""
    metaforas = []
    corresp_ont = []
    corresp_epi = []

    for met in results_list:
        id_expr = met["ID_expresion"]
        metaforas.append({
            "ID_expresion": id_expr,
            "ID_documento": met.get("ID_documento", ""),
            "ID_oracion": met.get("ID_oracion", ""),
            "pagina": met.get("pagina", ""),
            "expresion_metaforica": met.get("expresion_metaforica", ""),
            "contexto": met.get("contexto", ""),
            "foco": met.get("foco", ""),
            "foco_lemma": met.get("foco_lemma", ""),
            "foco_part_of_speech": met.get("foco_part_of_speech", ""),
            "significado_contextual": met.get("significado_contextual", ""),
            "significado_basico": met.get("significado_basico", ""),
            "dominio_fuente": met.get("dominio_fuente", ""),
            "dominio_meta": met.get("dominio_meta", ""),
            "metafora_conceptual": met.get("metafora_conceptual", ""),
            "enfoque": enfoque_label,
            "confianza_modelo": met.get("confianza_modelo", 1.0)
        })

        for co in met.get("correspondencias_ontologicas", []) or []:
            corresp_ont.append({
                "ID_correspondencia": f"CO-{id_expr}-{len(corresp_ont)+1:03d}",
                "ID_expresion": id_expr,
                "enfoque": enfoque_label,
                "elemento_fuente": co.get("elemento_fuente", ""),
                "elemento_meta": co.get("elemento_meta", ""),
                "evidencia_textual": co.get("evidencia_textual", "")
            })
        for ce in met.get("correspondencias_epistemicas", []) or []:
            corresp_epi.append({
                "ID_correspondencia_ep": f"CE-{id_expr}-{len(corresp_epi)+1:03d}",
                "ID_expresion": id_expr,
                "enfoque": enfoque_label,
                "tipo_inferencia": ce.get("tipo_inferencia", ""),
                "relacion_fuente": ce.get("relacion_fuente", ""),
                "inferencia_meta": ce.get("inferencia_meta", ""),
                "evidencia_textual": ce.get("evidencia_textual", "")
            })

    return pd.DataFrame(metaforas), pd.DataFrame(corresp_ont), pd.DataFrame(corresp_epi)


# Exportar cada enfoque que tenga resultados
resultados_ejecutados = {
    "claude": results_claude,
    "openai": results_openai,
    "huggingface": results_huggingface,
    "embeddings": results_embeddings,
    "groq_llama": results_groq_llama,
    "gemini": results_gemini,
}

correspondencias_ont_all = []
correspondencias_epi_all = []

for enfoque, results in resultados_ejecutados.items():
    if enfoque not in ENFOQUES_ACTIVOS:
        continue
    df_met, df_co, df_ce = flatten_approach_results(results, enfoque)

    path = os.path.join(OUTPUT_DIR, f"n1_metaforas_{enfoque}.csv")
    df_met.to_csv(path, index=False, encoding='utf-8-sig')
    print(f"✓ {path} ({len(df_met):,} filas)")

    if len(df_co) > 0:
        correspondencias_ont_all.append(df_co)
    if len(df_ce) > 0:
        correspondencias_epi_all.append(df_ce)

# Exportar muestra de oraciones procesadas
df_work_path = os.path.join(OUTPUT_DIR, "n1_muestra_oraciones.csv")
df_work.to_csv(df_work_path, index=False, encoding='utf-8-sig')
print(f"✓ {df_work_path} ({len(df_work):,} oraciones)")

# Correspondencias consolidadas
if correspondencias_ont_all:
    df_ont_total = pd.concat(correspondencias_ont_all, ignore_index=True)
    df_ont_total.to_csv(os.path.join(OUTPUT_DIR, "n1_correspondencias_ontologicas.csv"),
                         index=False, encoding='utf-8-sig')
    print(f"✓ n1_correspondencias_ontologicas.csv ({len(df_ont_total):,} filas)")

if correspondencias_epi_all:
    df_epi_total = pd.concat(correspondencias_epi_all, ignore_index=True)
    df_epi_total.to_csv(os.path.join(OUTPUT_DIR, "n1_correspondencias_epistemicas.csv"),
                         index=False, encoding='utf-8-sig')
    print(f"✓ n1_correspondencias_epistemicas.csv ({len(df_epi_total):,} filas)")

## 11. Carga de resultados desde disco

A partir de aquí, el notebook puede ejecutarse en sesiones posteriores **sin correr los enfoques de nuevo**. Esta celda lee los CSVs disponibles en `OUTPUT_DIR` según lo que esté en `ENFOQUES_A_VISUALIZAR`.

In [ ]:
# ============================================================
# 11. CARGA DESDE DISCO DE RESULTADOS POR ENFOQUE
# ============================================================

resultados_disponibles = {}
enfoques_no_encontrados = []

for enfoque in ENFOQUES_A_VISUALIZAR:
    path = os.path.join(OUTPUT_DIR, f"n1_metaforas_{enfoque}.csv")
    if os.path.exists(path):
        resultados_disponibles[enfoque] = pd.read_csv(path)
        print(f"✓ Cargado: n1_metaforas_{enfoque}.csv ({len(resultados_disponibles[enfoque]):,} filas)")
    else:
        enfoques_no_encontrados.append(enfoque)
        print(f"⚠ NO encontrado: n1_metaforas_{enfoque}.csv — será omitido de las visualizaciones")

# Actualizar lista efectiva de enfoques a visualizar (solo los disponibles)
ENFOQUES_DISPONIBLES = list(resultados_disponibles.keys())
print(f"\nEnfoques que SE visualizarán: {ENFOQUES_DISPONIBLES}")
if enfoques_no_encontrados:
    print(f"Enfoques solicitados pero faltantes: {enfoques_no_encontrados}")

if not ENFOQUES_DISPONIBLES:
    print("\n⚠ No hay datos disponibles para visualizar.")
    print("  Ejecuta primero algún enfoque o verifica ENFOQUES_A_VISUALIZAR.")

## 12. Comparación entre enfoques

Se calculan métricas de concordancia a nivel de oración (Cohen's kappa) para todos los pares de enfoques disponibles.

In [ ]:
# ============================================================
# 12. COMPARACIÓN ENTRE ENFOQUES
# ============================================================

def get_sentence_detection_vector(df, all_sents):
    """Vector binario: 1 si el enfoque detectó metáfora en esa oración, 0 si no."""
    detected = set(df["ID_oracion"].unique())
    return [1 if s in detected else 0 for s in all_sents]


if len(ENFOQUES_DISPONIBLES) >= 2:
    all_sents_set = set()
    for df in resultados_disponibles.values():
        all_sents_set.update(df["ID_oracion"].unique())
    # Si hay df_work disponible (modo ejecución), añade oraciones no detectadas
    try:
        all_sents_set.update(df_work["ID_oracion"].unique())
    except NameError:
        pass
    all_sents = sorted(all_sents_set)

    # Matriz de concordancia kappa
    kappa_matrix = pd.DataFrame(index=ENFOQUES_DISPONIBLES, columns=ENFOQUES_DISPONIBLES, dtype=float)
    comparison_rows = []
    for i, e1 in enumerate(ENFOQUES_DISPONIBLES):
        v1 = get_sentence_detection_vector(resultados_disponibles[e1], all_sents)
        for j, e2 in enumerate(ENFOQUES_DISPONIBLES):
            v2 = get_sentence_detection_vector(resultados_disponibles[e2], all_sents)
            if i == j:
                kappa_matrix.loc[e1, e2] = 1.0
            else:
                kappa_matrix.loc[e1, e2] = cohen_kappa_score(v1, v2)
            if i < j:
                # Solo pares únicos
                dom_fuente_concordancia = len(set(resultados_disponibles[e1]["dominio_fuente"].dropna()) &
                                              set(resultados_disponibles[e2]["dominio_fuente"].dropna()))
                met_conceptual_concordancia = len(set(resultados_disponibles[e1]["metafora_conceptual"].dropna()) &
                                                   set(resultados_disponibles[e2]["metafora_conceptual"].dropna()))
                comparison_rows.append({
                    "enfoque_A": e1,
                    "enfoque_B": e2,
                    "kappa_oracion": round(kappa_matrix.loc[e1, e2], 3),
                    "dominios_fuente_compartidos": dom_fuente_concordancia,
                    "metaforas_conceptuales_compartidas": met_conceptual_concordancia,
                    "costo_A": round(costos_enfoques.get(e1, 0.0), 2),
                    "costo_B": round(costos_enfoques.get(e2, 0.0), 2),
                    "tiempo_A": round(tiempos_enfoques.get(e1, 0.0), 1),
                    "tiempo_B": round(tiempos_enfoques.get(e2, 0.0), 1),
                })

    df_comparison = pd.DataFrame(comparison_rows)
    comp_path = os.path.join(OUTPUT_DIR, "n1_comparacion_enfoques.csv")
    df_comparison.to_csv(comp_path, index=False, encoding='utf-8-sig')
    print(f"✓ {comp_path}")
    print()
    print("RESUMEN DE COMPARACIÓN:")
    print("=" * 70)
    for _, row in df_comparison.iterrows():
        print(f"  {row['enfoque_A']} vs {row['enfoque_B']}: "
              f"kappa={row['kappa_oracion']:.3f}, "
              f"dom_fuente_comp={row['dominios_fuente_compartidos']}, "
              f"met_conc_comp={row['metaforas_conceptuales_compartidas']}")
else:
    print("⚠ Se necesitan al menos 2 enfoques disponibles para calcular concordancia.")
    kappa_matrix = None
    df_comparison = None

## 13. Consolidación: `n1_metaforas_primarias.csv`

Merge de todos los enfoques con un campo `confianza_cross_enfoques` que indica cuántos enfoques detectaron cada metáfora (basado en coincidencia de foco + oración).

In [ ]:
# ============================================================
# 13. CONSOLIDACIÓN DE TODOS LOS ENFOQUES
# ============================================================

if ENFOQUES_DISPONIBLES:
    dfs_consolidar = [resultados_disponibles[e] for e in ENFOQUES_DISPONIBLES]
    df_consolidado = pd.concat(dfs_consolidar, ignore_index=True)

    # Calcular confianza cross-enfoques: cuántos enfoques detectaron la misma combinación (oracion, foco)
    confianza_counter = Counter()
    for _, row in df_consolidado.iterrows():
        foco_key = row.get("foco", "") or row.get("foco_lemma", "") or row.get("expresion_metaforica", "")
        key = (row["ID_oracion"], foco_key.lower() if isinstance(foco_key, str) else "")
        confianza_counter[key] += 1

    # Deduplicar por (oracion, foco) para calcular cuántos enfoques únicos coincidieron
    unique_by_focus = df_consolidado.groupby(["ID_oracion", "foco"])["enfoque"].nunique().to_dict()

    def get_confianza(row):
        key = (row["ID_oracion"], row.get("foco", ""))
        return unique_by_focus.get(key, 1)

    df_consolidado["confianza_cross_enfoques"] = df_consolidado.apply(get_confianza, axis=1)

    path_consolidado = os.path.join(OUTPUT_DIR, "n1_metaforas_primarias.csv")
    df_consolidado.to_csv(path_consolidado, index=False, encoding='utf-8-sig')
    print(f"✓ {path_consolidado} ({len(df_consolidado):,} filas)")
    print(f"  Enfoques incluidos: {ENFOQUES_DISPONIBLES}")
    print(f"  Distribución de confianza cross-enfoques:")
    print(df_consolidado["confianza_cross_enfoques"].value_counts().sort_index().to_string())
else:
    print("⚠ No hay enfoques disponibles para consolidar.")
    df_consolidado = pd.DataFrame()

In [ ]:
# ============================================================
# 13B. METADATOS DE EJECUCIÓN
# ============================================================

import datetime
import importlib

metadata = {
    "nivel": "N1",
    "timestamp": datetime.datetime.now().isoformat(),
    "enfoques_activos_configurados": ENFOQUES_ACTIVOS,
    "enfoques_a_visualizar_configurados": ENFOQUES_A_VISUALIZAR,
    "enfoques_disponibles_en_disco": ENFOQUES_DISPONIBLES,
    "modo": "STANDALONE" if STANDALONE_MODE else f"HYBRID_{HYBRID_LLM}",
    "modelos": {
        "claude": CLAUDE_MODEL,
        "openai": OPENAI_MODEL,
        "huggingface": HF_MODEL,
        "embeddings": EMBEDDING_MODEL,
        "groq_llama": GROQ_MODEL,
        "gemini": GEMINI_MODEL,
    },
    "sample_mode": SAMPLE_MODE,
    "sample_size": SAMPLE_SIZE,
    "temperatura_llm": TEMPERATURE,
    "embedding_contrast_threshold": EMBEDDING_CONTRAST_THRESHOLD,
    "oraciones_procesadas": len(df_work) if 'df_work' in dir() else "N/A",
    "metaforas_por_enfoque": {e: len(df) for e, df in resultados_disponibles.items()},
    "tiempos_seg": {e: round(t, 1) for e, t in tiempos_enfoques.items()},
    "costos_usd": {e: round(c, 2) for e, c in costos_enfoques.items()},
    "costo_total_usd": round(sum(costos_enfoques.values()), 2),
}

with open(os.path.join(OUTPUT_DIR, "n1_metadata.json"), 'w', encoding='utf-8') as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)
print(f"✓ n1_metadata.json")

## 15. Muestra para evaluación human-in-the-loop

Muestra balanceada de 30-50 metáforas para revisión manual. Se toma un número proporcional de cada enfoque disponible.

In [ ]:
# ============================================================
# 15. MUESTRA BALANCEADA PARA EVALUACIÓN HUMANA
# ============================================================

SAMPLE_PER_ENFOQUE = 15

if ENFOQUES_DISPONIBLES:
    muestras = []
    for enfoque in ENFOQUES_DISPONIBLES:
        df_sub = resultados_disponibles[enfoque]
        if len(df_sub) > 0:
            n = min(SAMPLE_PER_ENFOQUE, len(df_sub))
            muestras.append(df_sub.sample(n=n, random_state=42))

    if muestras:
        df_eval = pd.concat(muestras, ignore_index=True)
        cols_to_keep = [c for c in [
            "ID_expresion", "enfoque", "contexto", "expresion_metaforica", "foco",
            "foco_part_of_speech", "significado_contextual", "significado_basico",
            "dominio_fuente", "dominio_meta", "metafora_conceptual", "confianza_modelo"
        ] if c in df_eval.columns]
        df_eval = df_eval[cols_to_keep].copy()

        df_eval["es_metafora_correcta"] = ""
        df_eval["componentes_correctos"] = ""
        df_eval["observaciones"] = ""

        eval_path = os.path.join(OUTPUT_DIR, "n1_muestra_evaluacion.csv")
        df_eval.to_csv(eval_path, index=False, encoding='utf-8-sig')
        print(f"✓ {eval_path} ({len(df_eval)} metáforas balanceadas)")
        print(f"  Enfoques muestreados: {ENFOQUES_DISPONIBLES}")
        print(f"  Muestras por enfoque: hasta {SAMPLE_PER_ENFOQUE}")
else:
    print("⚠ Sin enfoques disponibles para generar muestra de evaluación.")

## 16. Resumen final

In [ ]:
# ============================================================
# 16. RESUMEN FINAL
# ============================================================

print("=" * 70)
print("N1 — METÁFORAS PRIMARIAS COMPLETADO")
print("=" * 70)
print()
print(f"Configuración:")
print(f"  Enfoques ACTIVOS en esta corrida:  {ENFOQUES_ACTIVOS}")
print(f"  Enfoques DISPONIBLES en disco:     {ENFOQUES_DISPONIBLES}")
print(f"  Modo:                              {'STANDALONE' if STANDALONE_MODE else f'HÍBRIDO ({HYBRID_LLM})'}")
print(f"  Oraciones procesadas:              {len(df_work) if 'df_work' in dir() else 'N/A (carga desde disco)'}")
print()
print(f"Resultados por enfoque:")
for enfoque in ENFOQUES_DISPONIBLES:
    n = len(resultados_disponibles[enfoque])
    tiempo = tiempos_enfoques.get(enfoque, 0)
    costo = costos_enfoques.get(enfoque, 0.0)
    print(f"  {enfoque:15s}: {n:5d} metáforas | {tiempo:6.1f}s | ${costo:6.2f} USD")
print()
print(f"Costo total APIs: ${sum(costos_enfoques.values()):.2f} USD")
print()
print(f"Archivos generados en {OUTPUT_DIR}:")
for f in sorted(os.listdir(OUTPUT_DIR)):
    fp = os.path.join(OUTPUT_DIR, f)
    size = os.path.getsize(fp) / 1024
    icon = "📊" if f.endswith(".html") else "🖼" if f.endswith(".png") else "📄"
    print(f"  {icon} {f}  ({size:.0f} KB)")
print()
print("➡ SIGUIENTE: Ejecutar N1_metafora_primaria_viz.ipynb para visualizaciones
➡ LUEGO: Ejecutar N2_convencionalizacion.ipynb")
print("  Entrada requerida: n1_metaforas_primarias.csv")